# RQ2 - Dynamic Adaptation reaction

Plots switch timing, monitored states, and dynamic fault-window context.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'RQ2').exists() and (ROOT / 'Paper Results').exists():
    ROOT = ROOT / 'Paper Results'

rq2 = ROOT / 'RQ2'
timeseries_path = rq2 / 'adaptation_state_timeseries.csv'
switch_path = rq2 / 'switch_timeline.csv'
summary_path = rq2 / 'summary_by_profile.csv'

if not timeseries_path.exists():
    print(f'RQ2 data not found: {timeseries_path}')
    print('Run: python "Paper Results/run_rq2_experiments.py"')
else:
    states = pd.read_csv(timeseries_path)
    switches = pd.read_csv(switch_path) if switch_path.exists() else pd.DataFrame()
    summary = pd.read_csv(summary_path) if summary_path.exists() else pd.DataFrame()

    numeric_cols = [
        'sim_time', 'sim_day', 'sim_hour', 'performance_score', 'latency',
        'pending_ratio', 'failure_rate', 'message_loss_rate',
        'score_latency', 'score_pending', 'score_failure', 'score_message_loss'
    ]
    for col in numeric_cols:
        if col in states:
            states[col] = pd.to_numeric(states[col], errors='coerce')
    if not switches.empty:
        for col in ['sim_time', 'sim_day', 'sim_hour', 'performance_score']:
            if col in switches:
                switches[col] = pd.to_numeric(switches[col], errors='coerce')

    first_runs = states.sort_values(['profile', 'run_id']).groupby('profile')['run_id'].first()
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))

    ax = axes[0, 0]
    for profile, run_id in first_runs.items():
        data = states[states['run_id'] == run_id].sort_values('sim_time')
        ax.plot(data['sim_time'] / 86400, data['performance_score'], label=profile, linewidth=1.6)
    ax.axhline(0.35, color='#777777', linestyle='--', linewidth=1, label='low threshold')
    ax.axhline(0.65, color='#333333', linestyle='--', linewidth=1, label='high threshold')
    ax.set_title('Monitored score over time')
    ax.set_xlabel('Simulation day')
    ax.set_ylabel('S(W)')
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)

    ax = axes[0, 1]
    if switches.empty:
        ax.text(0.5, 0.5, 'No switches recorded', ha='center', va='center')
        ax.set_axis_off()
    else:
        pivot = switches.pivot_table(index='profile', columns='selected_coord_label', values='switch_idx', aggfunc='count', fill_value=0)
        pivot.plot(kind='bar', stacked=True, ax=ax, color=['#4C78A8', '#F2B701', '#E45756', '#54A24B'])
        ax.set_title('Switches by selected mechanism')
        ax.set_xlabel('Profile')
        ax.set_ylabel('Switch count')
        ax.tick_params(axis='x', rotation=25)
        ax.grid(axis='y', alpha=0.25)

    ax = axes[1, 0]
    state_counts = states.pivot_table(index='profile', columns='performance_state', values='sim_time', aggfunc='count', fill_value=0)
    state_frac = state_counts.div(state_counts.sum(axis=1), axis=0)
    state_frac.plot(kind='bar', stacked=True, ax=ax, color=['#E45756', '#F2B701', '#54A24B', '#999999'])
    ax.set_title('Fraction of adaptation checks per state')
    ax.set_xlabel('Profile')
    ax.set_ylabel('Fraction')
    ax.tick_params(axis='x', rotation=25)
    ax.grid(axis='y', alpha=0.25)

    ax = axes[1, 1]
    component_cols = [c for c in ['score_latency', 'score_pending', 'score_failure', 'score_message_loss'] if c in states]
    if component_cols:
        comp = states.groupby('phase_label')[component_cols].mean().sort_index()
        comp.plot(kind='bar', ax=ax, color=['#4C78A8', '#F2B701', '#E45756', '#72B7B2'])
        ax.set_title('Mean score components by phase')
        ax.set_xlabel('Phase')
        ax.set_ylabel('Normalized contribution')
        ax.tick_params(axis='x', rotation=25)
        ax.grid(axis='y', alpha=0.25)
    else:
        ax.text(0.5, 0.5, 'No score components available', ha='center', va='center')
        ax.set_axis_off()

    fig.suptitle('RQ2: DA reaction to evolving operating conditions', fontsize=14)
    fig.tight_layout()
    plt.show()

    if not summary.empty:
        display(summary)
